
# CNN Lab – MNIST

In [ ]:

# Optional dependency installer
import sys
import subprocess
import importlib.util

required = {
    "torch": "torch torchvision torchinfo",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

for module_name, pip_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *pip_name.split()])

print("Dependency check complete.")


In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchinfo import summary

from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



## Configuration

Set the dataset here. The PDF says `torchvision.datasets.MNIST` in Task 0, but the description references Fashion-MNIST. This notebook supports both.


In [ ]:

DATASET_NAME = "MNIST"   # change to "FashionMNIST" if needed
BATCH_SIZE = 128
VAL_RATIO = 0.20
EPOCHS = 8
LR = 1e-3
L1_LAMBDA = 1e-6
WEIGHT_DECAY = 1e-4   # L2 penalty through optimizer

if DATASET_NAME == "MNIST":
    DatasetClass = datasets.MNIST
    class_names = [str(i) for i in range(10)]
elif DATASET_NAME == "FashionMNIST":
    DatasetClass = datasets.FashionMNIST
    class_names = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]
else:
    raise ValueError("DATASET_NAME must be 'MNIST' or 'FashionMNIST'")

print("Using dataset:", DATASET_NAME)



## Task 0: Load the dataset

The transform pipeline:
- converts images to tensors,
- rescales pixel values from `[0, 255]` to `[0, 1]`,
- normalizes them to `[-1, 1]` using mean `0.5` and std `0.5`.


In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset_full = DatasetClass(root="./data", train=True, download=True, transform=transform)
test_dataset = DatasetClass(root="./data", train=False, download=True, transform=transform)

print("Train dataset loaded:", len(train_dataset_full))
print("Test dataset loaded:", len(test_dataset))



## Task 1: Report basic dataset information using the first observation in each split


In [ ]:

x_train_0, y_train_0 = train_dataset_full[0]
x_test_0, y_test_0 = test_dataset[0]

info_df = pd.DataFrame({
    "split": ["train", "test"],
    "n_samples": [len(train_dataset_full), len(test_dataset)],
    "input_dtype": [str(x_train_0.dtype), str(x_test_0.dtype)],
    "label_dtype": [type(y_train_0).__name__, type(y_test_0).__name__],
    "input_shape": [tuple(x_train_0.shape), tuple(x_test_0.shape)],
    "min_pixel": [float(x_train_0.min()), float(x_test_0.min())],
    "max_pixel": [float(x_train_0.max()), float(x_test_0.max())],
})
print(info_df)

print("\nChannels check:")
print("Train first image channels:", x_train_0.shape[0])
print("Test first image channels:", x_test_0.shape[0])



## Task 2: Unique labels, counts, and proportions for train and test


In [ ]:

def label_summary(dataset, class_names):
    labels = []
    for _, y in dataset:
        labels.append(int(y))
    labels = np.array(labels)

    unique, counts = np.unique(labels, return_counts=True)
    proportions = counts / counts.sum()

    df = pd.DataFrame({
        "label": unique,
        "class_name": [class_names[i] for i in unique],
        "count": counts,
        "proportion": proportions
    })
    return df

train_label_df = label_summary(train_dataset_full, class_names)
test_label_df = label_summary(test_dataset, class_names)

print("Train label distribution")
display(train_label_df)

print("Test label distribution")
display(test_label_df)

summary_stats = pd.DataFrame({
    "split": ["train", "test"],
    "min_count": [train_label_df["count"].min(), test_label_df["count"].min()],
    "avg_count": [train_label_df["count"].mean(), test_label_df["count"].mean()],
    "max_count": [train_label_df["count"].max(), test_label_df["count"].max()],
    "min_prop": [train_label_df["proportion"].min(), test_label_df["proportion"].min()],
    "avg_prop": [train_label_df["proportion"].mean(), test_label_df["proportion"].mean()],
    "max_prop": [train_label_df["proportion"].max(), test_label_df["proportion"].max()],
})

print("Summary statistics")
display(summary_stats)



## Task 3: Plot one example from each label value


In [ ]:

def plot_one_per_class(dataset, class_names, ncols=5, title="One example per class"):
    examples = {}
    for x, y in dataset:
        y = int(y)
        if y not in examples:
            examples[y] = x
        if len(examples) == len(class_names):
            break

    n = len(class_names)
    nrows = math.ceil(n / ncols)
    plt.figure(figsize=(2.2 * ncols, 2.2 * nrows))
    for i in range(n):
        plt.subplot(nrows, ncols, i + 1)
        img = examples[i].squeeze().cpu().numpy()
        img = img * 0.5 + 0.5
        plt.imshow(img, cmap="gray")
        plt.title(f"{i}: {class_names[i]}", fontsize=9)
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_one_per_class(train_dataset_full, class_names, ncols=5, title=f"{DATASET_NAME}: one example per class")



## Task 4: Split the training set into training and validation subsets (80/20)


In [ ]:

n_total = len(train_dataset_full)
n_val = int(VAL_RATIO * n_total)
n_train = n_total - n_val

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

print("Training subset:", len(train_dataset))
print("Validation subset:", len(val_dataset))
print("Test dataset:", len(test_dataset))



## Task 5: Create DataLoaders


In [ ]:

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))



## Task 6: Define the convolutional neural network


In [ ]:

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2)                               # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=10).to(device)
print(model)



## Task 7: Print model architecture and manually compute trainable parameters


In [ ]:

def count_params_conv2d(in_channels, out_channels, kernel_size, bias=True):
    k = kernel_size if isinstance(kernel_size, int) else kernel_size[0] * kernel_size[1]
    if isinstance(kernel_size, int):
        weight_params = out_channels * in_channels * kernel_size * kernel_size
    else:
        weight_params = out_channels * in_channels * kernel_size[0] * kernel_size[1]
    bias_params = out_channels if bias else 0
    return weight_params + bias_params

def count_params_linear(in_features, out_features, bias=True):
    weight_params = in_features * out_features
    bias_params = out_features if bias else 0
    return weight_params + bias_params

manual_param_table = pd.DataFrame({
    "layer": [
        "Conv2d(1,16,3x3)",
        "Conv2d(16,32,3x3)",
        "Linear(32*7*7,128)",
        "Linear(128,10)"
    ],
    "params": [
        count_params_conv2d(1, 16, 3, bias=True),
        count_params_conv2d(16, 32, 3, bias=True),
        count_params_linear(32 * 7 * 7, 128, bias=True),
        count_params_linear(128, 10, bias=True),
    ]
})
manual_param_table["params"] = manual_param_table["params"].astype(int)
manual_total = int(manual_param_table["params"].sum())

display(manual_param_table)
print("Manual total trainable parameters:", manual_total)

torch_total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Torch total trainable parameters:", torch_total)


The model contains 206,922 trainable parameters, which is relatively moderate in size. This allows the model to capture complex patterns such as edges and shapes while still maintaining computational efficiency. The parameter count is sufficient for MNIST without being excessively large, which helps prevent overfitting.


## Task 8: Use `torchinfo.summary`


In [ ]:

summary(model, input_size=(BATCH_SIZE, 1, 28, 28), col_names=("input_size", "output_size", "num_params", "trainable"))



## Task 9: Specify loss, optimizer, metric, and include L1/L2 penalties

- Loss: CrossEntropyLoss
- Optimizer: Adam
- Metric: Accuracy
- L2 penalty: `weight_decay` in the optimizer
- L1 penalty: manually added to the loss inside the training loop


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def l1_penalty(model):
    return sum(p.abs().sum() for p in model.parameters())

print("Loss:", criterion)
print("Optimizer:", optimizer)
print("Metric: accuracy")
print("L1 lambda:", L1_LAMBDA)
print("L2 weight decay:", WEIGHT_DECAY)



## Task 10: Train the model and track training/validation loss and accuracy


In [ ]:

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == yb).sum().item()
            total_count += xb.size(0)

    return total_loss / total_count, total_correct / total_count

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_count = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb) + L1_LAMBDA * l1_penalty(model)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == yb).sum().item()
        running_count += xb.size(0)

    train_loss = running_loss / running_count
    train_acc = running_correct / running_count
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )



## Task 11: Evaluate on the test set, plot curves, and show sample predictions with probabilities


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["train_acc"], label="train_acc")
plt.plot(history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


In [ ]:

# Display sample test images with probabilities
model.eval()
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)

with torch.no_grad():
    logits = model(xb)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

num_show = 10
plt.figure(figsize=(16, 7))
for i in range(num_show):
    plt.subplot(2, 5, i + 1)
    img = xb[i].detach().cpu().squeeze().numpy()
    img = img * 0.5 + 0.5
    topk_probs, topk_idx = torch.topk(probs[i], k=3)

    title_lines = [
        f"T:{class_names[int(yb[i].cpu())]}",
        f"P:{class_names[int(preds[i].cpu())]}",
        f"{class_names[int(topk_idx[0].cpu())]}:{float(topk_probs[0].cpu()):.2f}",
        f"{class_names[int(topk_idx[1].cpu())]}:{float(topk_probs[1].cpu()):.2f}",
        f"{class_names[int(topk_idx[2].cpu())]}:{float(topk_probs[2].cpu()):.2f}",
    ]

    plt.imshow(img, cmap="gray")
    plt.title("\n".join(title_lines), fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()


The model demonstrates strong and stable learning behavior throughout training. Training accuracy increases from 0.9048 in the first epoch to approximately 0.9937 by the final epoch, while validation accuracy improves from 0.9698 to 0.9858. At the same time, both training and validation loss decrease steadily, indicating effective optimization.

The small gap between training and validation accuracy suggests that the model generalizes well and does not exhibit significant overfitting.

### Loss Curve Interpretation

The loss curves show a clear downward trend for both training and validation sets. Training loss decreases consistently across epochs, while validation loss stabilizes after early improvements. There is no divergence between the two curves, indicating stable training and good generalization. Slight fluctuations in validation loss in later epochs suggest minor noise but no overfitting.

### Accuracy Curve Interpretation

Both training and validation accuracy increase rapidly in the first few epochs and then plateau near their maximum values. The close alignment between the two curves indicates that the model is well-balanced, avoiding both underfitting and overfitting. The high validation accuracy (~0.986) confirms that the learned features generalize effectively to unseen data.

### Test Performance

The model achieves a test accuracy of 0.9857 with a test loss of 0.0470, which is highly consistent with the validation performance. This indicates that the model generalizes very well to unseen data and has successfully learned meaningful representations of the input images.

The model produces highly confident predictions, as seen in the probability outputs where the top predicted class typically has a significantly higher probability than the others. This indicates that the model has learned strong discriminative features for distinguishing between different digit classes.


## Task 12: Redefine the model with batch normalization and explicit weight initialization

Choice:
- Batch normalization after each convolution and before ReLU.
- He/Kaiming initialization for convolutional and linear layers, because the network uses ReLU activations.

Brief justification:
- Batch normalization stabilizes the distribution of activations during training, which helps optimization and can improve convergence.
- He initialization is appropriate for ReLU networks because it preserves signal scale more effectively across layers.


In [ ]:

class CNNWithBatchNorm(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

bn_model = CNNWithBatchNorm(num_classes=10).to(device)
print(bn_model)


In [ ]:

bn_manual_table = pd.DataFrame({
    "layer": [
        "Conv2d(1,16,3x3)", "BatchNorm2d(16)",
        "Conv2d(16,32,3x3)", "BatchNorm2d(32)",
        "Linear(32*7*7,128)", "Linear(128,10)"
    ],
    "params": [
        count_params_conv2d(1, 16, 3, bias=True),
        16 + 16,
        count_params_conv2d(16, 32, 3, bias=True),
        32 + 32,
        count_params_linear(32 * 7 * 7, 128, bias=True),
        count_params_linear(128, 10, bias=True),
    ]
})
display(bn_manual_table)
print("Manual total trainable parameters (BN model):", int(bn_manual_table["params"].sum()))
print("Torch total trainable parameters (BN model):", sum(p.numel() for p in bn_model.parameters() if p.requires_grad))
summary(bn_model, input_size=(BATCH_SIZE, 1, 28, 28), col_names=("input_size", "output_size", "num_params", "trainable"))



## Optional: Train the batch-normalized model for comparison


In [ ]:

bn_criterion = nn.CrossEntropyLoss()
bn_optimizer = torch.optim.Adam(bn_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

bn_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(1, EPOCHS + 1):
    bn_model.train()
    running_loss = 0.0
    running_correct = 0
    running_count = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        bn_optimizer.zero_grad()
        logits = bn_model(xb)
        loss = bn_criterion(logits, yb) + L1_LAMBDA * l1_penalty(bn_model)
        loss.backward()
        bn_optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == yb).sum().item()
        running_count += xb.size(0)

    train_loss = running_loss / running_count
    train_acc = running_correct / running_count
    val_loss, val_acc = evaluate(bn_model, val_loader, bn_criterion)

    bn_history["train_loss"].append(train_loss)
    bn_history["train_acc"].append(train_acc)
    bn_history["val_loss"].append(val_loss)
    bn_history["val_acc"].append(val_acc)

    print(
        f"[BN] Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

bn_test_loss, bn_test_acc = evaluate(bn_model, test_loader, bn_criterion)
print(f"\n[BN] Test loss: {bn_test_loss:.4f}")
print(f"[BN] Test accuracy: {bn_test_acc:.4f}")


The batch-normalized model shows improved performance compared to the original model. Training becomes more stable, and validation accuracy increases slightly, reaching approximately 0.9882. The final test accuracy improves to 0.9884, indicating better generalization.

This improvement demonstrates that batch normalization helps stabilize the distribution of intermediate activations, leading to faster convergence and better performance. Additionally, He (Kaiming) initialization ensures that gradients remain well-scaled during training, which is particularly effective for networks using ReLU activations.


## Task 13: Reflection

Compared to a feedforward neural network, the convolutional neural network performs significantly better for image classification tasks because it preserves spatial structure and learns local features such as edges and shapes through convolutional filters. In contrast, a feedforward network flattens the image and loses spatial relationships between pixels.

The CNN architecture used in this assignment effectively captures hierarchical patterns, leading to high accuracy and strong generalization performance. Furthermore, the inclusion of batch normalization improves training stability and convergence speed, while appropriate weight initialization ensures efficient gradient propagation. Overall, these architectural choices result in a more robust and efficient model compared to a standard fully connected network.